# RelCal robustness vs multipole binning and lmax
This notebook is similar in spirit to `mk_relcal_compare.ipynb`, but focuses on RelCal stability when changing multipole bin width and $\ell_{\max}$.
It builds temporary config variants, reruns RelCal, and reports pairwise and network-level shifts relative to a baseline setup.

In [1]:
import os
for variable in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS',
                 'NUMEXPR_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    os.environ[variable] = '1'

%cd /global/homes/l/lonappan/workspace/cosmic_birefringence
%matplotlib inline

from copy import deepcopy
from pathlib import Path
import numpy as np
import yaml

from cosmic_bire.relcal import RelCal

base_config = Path('configs/planck_hfi.yml')
mask = '0'
ref = '353B'

with base_config.open('r') as f:
    base_cfg = yaml.safe_load(f)

variant_dir = Path('notebooks/_tmp_relcal_variants')
variant_dir.mkdir(parents=True, exist_ok=True)

print('base config:', base_config)
print('variant dir:', variant_dir.resolve())

/global/u2/l/lonappan/workspace/cosmic_birefringence
base config: configs/planck_hfi.yml
variant dir: /global/u2/l/lonappan/workspace/cosmic_birefringence/notebooks/_tmp_relcal_variants


In [2]:
# Choose settings where (lmax - lmin) is divisible by bin_width
settings = [
    {'tag': 'base',          'lmax': 1491, 'bin_width': 20},
    {'tag': 'dl10_lmax1491', 'lmax': 1491, 'bin_width': 10},
    {'tag': 'dl30_lmax1491', 'lmax': 1491, 'bin_width': 30},
    {'tag': 'dl20_lmax1411', 'lmax': 1411, 'bin_width': 20},
    {'tag': 'dl20_lmax1231', 'lmax': 1231, 'bin_width': 20},
]

def write_variant_config(tag, lmax, bin_width):
    cfg = deepcopy(base_cfg)
    lmin = int(cfg['likelihood'].get('lmin', 51))
    if (lmax - lmin) % bin_width != 0:
        raise ValueError(f'Invalid combo for {tag}: (lmax-lmin) % bin_width != 0')
    cfg['likelihood']['lmax'] = int(lmax)
    cfg['likelihood']['bin_width'] = int(bin_width)
    out = variant_dir / f'planck_hfi_{tag}.yml'
    with out.open('w') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    return out

results = {}
for s in settings:
    cfg_path = write_variant_config(s['tag'], s['lmax'], s['bin_width'])
    rc = RelCal(str(cfg_path), mask=mask)
    pair_fits = rc.fit_all_pairs()
    alpha_rel, cov_rel, _ = rc.fit_network(ref=ref)
    err_rel = np.sqrt(np.abs(np.diag(cov_rel)))
    labels = [rc.map_label(k) for k in range(rc.nmaps)]
    results[s['tag']] = {
        'config': cfg_path,
        'lmax': s['lmax'],
        'bin_width': s['bin_width'],
        'labels': labels,
        'pair_fits': pair_fits,
        'alpha_rel': alpha_rel,
        'err_rel': err_rel,
    }
    print(f"{s['tag']:14s} lmax={s['lmax']} dl={s['bin_width']} bins={rc.n_bins}")

base           lmax=1491 dl=20 bins=72
dl10_lmax1491  lmax=1491 dl=10 bins=144
dl30_lmax1491  lmax=1491 dl=30 bins=48
dl20_lmax1411  lmax=1411 dl=20 bins=68
dl20_lmax1231  lmax=1231 dl=20 bins=59


In [3]:
baseline = results['base']

print('=== Pairwise robustness relative to baseline (mask=0) ===')
print(f"{'setting':14s} {'pair':12s} {'base Dalpha':>14s} {'test Dalpha':>14s} {'diff':>10s} {'sigma shift':>12s}")

all_rows = []
for tag, data in results.items():
    if tag == 'base':
        continue
    for pair, (d_test, s_test) in data['pair_fits'].items():
        d_base, s_base = baseline['pair_fits'][pair]
        diff = d_test - d_base
        denom = np.sqrt(s_test**2 + s_base**2)
        z = np.nan if np.isclose(denom, 0.0) else diff / denom
        all_rows.append((tag, f"{pair[0]}x{pair[1]}", d_base, d_test, diff, z))

all_rows.sort(key=lambda r: 0.0 if np.isnan(r[5]) else -abs(r[5]))
for tag, pair_name, d_base, d_test, diff, z in all_rows[:30]:
    ztxt = '   n/a' if np.isnan(z) else f"{z:+9.2f}\u03c3"
    print(f"{tag:14s} {pair_name:12s} {d_base:+10.4f} {d_test:+10.4f} {diff:+10.4f} {ztxt:>12s}")

=== Pairwise robustness relative to baseline (mask=0) ===
setting        pair            base Dalpha    test Dalpha       diff  sigma shift
dl30_lmax1491  217Ax353B       +0.1653    +0.1713    +0.0061       +0.12σ
dl20_lmax1231  143Ax217A       +0.1359    +0.1261    -0.0099       -0.11σ
dl30_lmax1491  353Ax353B       +0.0355    +0.0390    +0.0035       +0.11σ
dl10_lmax1491  100Ax100B       +0.1051    +0.0838    -0.0213       -0.10σ
dl30_lmax1491  353Ax217B       -0.1074    -0.1030    +0.0044       +0.10σ
dl30_lmax1491  100Ax100B       +0.1051    +0.0856    -0.0196       -0.09σ
dl10_lmax1491  100Ax143A       -0.3066    -0.3217    -0.0151       -0.09σ
dl30_lmax1491  353Ax143B       -0.4219    -0.4144    +0.0075       +0.08σ
dl30_lmax1491  217Ax217B       +0.0618    +0.0659    +0.0041       +0.08σ
dl10_lmax1491  353Ax100B       +0.3727    +0.3566    -0.0161       -0.07σ
dl30_lmax1491  143Ax353B       +0.2360    +0.2432    +0.0071       +0.07σ
dl20_lmax1231  100Ax217A       -0.3552    -0.3

In [4]:
print('=== Network-fit robustness relative to baseline (alpha - alpha_ref) ===')
print(f"{'setting':14s} {'RMS shift':>12s} {'max |shift|':>12s} {'at map':>8s}")

a0 = baseline['alpha_rel']
labels0 = baseline['labels']

for tag, data in results.items():
    if tag == 'base':
        continue
    a = data['alpha_rel']
    shift = a - a0
    rms = np.sqrt(np.mean(shift**2))
    imax = int(np.argmax(np.abs(shift)))
    print(f"{tag:14s} {rms:12.4f} {shift[imax]:+12.4f} {labels0[imax]:>8s}")

=== Network-fit robustness relative to baseline (alpha - alpha_ref) ===
setting           RMS shift  max |shift|   at map
dl10_lmax1491        0.0057      +0.0131     100B
dl30_lmax1491        0.0048      +0.0106     100B
dl20_lmax1411        0.0007      -0.0012     217A
dl20_lmax1231        0.0040      -0.0078     100B


## Interpretation notes
- Small sigma shifts across settings indicate robust relative-angle recovery.
- If only high-frequency channels move when reducing $\ell_{\max}$, that usually indicates high-$\ell$ sensitivity rather than a global calibration failure.
- Prioritize stability in same-frequency A$\times$B pairs; large shifts there can flag residual systematics or covariance mismatch.